# Data Cleaning

This notebook validates all logbook entries after they are initialy downloaded from our Database.

## Table of Contents

0. [Directory Setup](#0.-Directory-Setup)
1. [Import Data](#1.-Import-Data)
2. [Initial Cleanup](#2.-Initial-Cleanup)
    - 2.1 [Drop Test Entries & Problem Logbooks](#2.1-Drop-Test-Entries-&-Problem-Logbooks)
    - 2.2 [DateTime Formatting](#2.2-DateTime-Formatting)
3. [Remove Duplicate Entries](#3.-Remove-Duplicate-Entries)
    - 3.1 [Known Duplicate Logbooks](#3.1-Known-Duplicate-Logbooks)
    - 3.2 [Double Dates](#3.2-Double-Dates)
4. [Clean Text Columns](#4.-Clean-Text-Columns)
    - 4.1 [Page, Depth, and Ship Sightings](#4.1-Page,-Depth,-and-Ship-Sightings)
    - 4.2 [Common String Cleanup](#4.2-Common-String-Cleanup)
5. [Standardize Wind Data](#5.-Clean-Wind-Data)
    - 5.1 [Wind Direction](#5.1-Wind-Direction)
    - 5.2 [Wind Speed/Force](#5.2-Wind-Speed/Force)
    - 5.3 [Sea State, Cloud Cover, and Weather](#5.3-Sea-State,-Cloud-Cover,-and-Weather)
6. [Coordinate Validation](#6.-Coordinate-Cleaning)
    - 6.1 [Format Normalization & Conversion](#6.1-Format-Normalization-&-Conversion)
    - 6.2 [Unrealistic Coordinates](#6.2-Unrealistic-Coordinates)
7. [Visual Inspection](#7.-Visual-Inspection)
8. [Export & Final Standardization](#8.-Export-&-Final-Standardization)
    - 8.1 [Save Cleaned Dataset](#8.1-Save-Cleaned-Dataset)
    - 8.2 [Standardize Beaufort Classifications](#8.2-Standardize-Beaufort-Classifications)

## 0. Directory Setup
Imports and folder paths.

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime, os
from cartopy import crs as ccrs
from cartopy import feature as cfeature
import sys
import re
from IPython.display import clear_output
import datetime
import textwrap
import csv

#set defaults
pd.options.display.max_columns = 50
print("Last updated on {}".format(datetime.datetime.now().ctime()))

Last updated on Tue Jun 16 14:48:22 2026


In [2]:
# Setting up a folder system to save files and figures to 
# Create folders called "Figures", "Text Files", and "CSV Files" manually to save everything to 

# Get current directory
current_directory = os.getcwd()
#print(current_directory)

# Specify the path to the folder you want to save data to 
Figures = os.path.join(current_directory, 'manuscript_figures')
Files = os.path.join(current_directory, 'output_txt_files')
CSV = os.path.join(current_directory, 'csv_files')
PKL = os.path.join(current_directory, "pkl_files")

#NOTE: copy and past permanent_txt_files from previous export
# Create directories only if they don't exist
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

## 1. Import Data

In [3]:
# labels corresponding to missing data
na_values = ['No observation', 'No observations', 'No Observation', 'No Observations',
             'no observation', 'no observations', 'None given', 'none given', 'None Given', 'none',
             'none recorded', 'not recorded', 'None recorded', 'Not given', 'not given', ' ', 'N.A.', 
             'Na', 'Does not say', 'N.A', 'Deos not say', 'N A', 'NA']

export_csv = 'logentries-export-2026-06-16.csv'

og_df = pd.read_csv(os.path.join(CSV, export_csv), na_values=na_values, low_memory=False)

# Combine using column names
og_df['DateTime'] = pd.to_datetime(og_df['Entry Date'] + ' ' + og_df['Local Time'], errors='coerce')
og_df.columns

Index(['ID', 'LogBook ID', 'Bottom', 'Cloud Cover', 'Depth', 'Depth Unit',
       'Entry Date', 'Landmark', 'Latitude', 'Local Time', 'Longitude', 'Page',
       'Sea State', 'Ship Heading/Course', 'Ship Sightings', 'Wind Direction',
       'Wind Speed/Force', 'DateTime'],
      dtype='str')

In [4]:
# Dropping columns not needed
columns_to_drop = ['Current', '2. Ship Heading/Course', '2. Wind Direction', '2. Wind Speed/Force', 
                   '2. Sea State', '2. Cloud Cover', '2. Weather', '3. Ship Heading', '3. Wind Direction', 
                   '3. Wind Speed/Force', '3. Sea State', '3. Cloud Cover', '3. Weather']

# Drop the specified columns
trimmed_df = og_df.copy()
for to_drop in columns_to_drop:
    try:
        trimmed_df.drop(columns=columns_to_drop, inplace=True)
    except KeyError:
        print(f"'{to_drop}' not found, skipping.")

'Current' not found, skipping.
'2. Ship Heading/Course' not found, skipping.
'2. Wind Direction' not found, skipping.
'2. Wind Speed/Force' not found, skipping.
'2. Sea State' not found, skipping.
'2. Cloud Cover' not found, skipping.
'2. Weather' not found, skipping.
'3. Ship Heading' not found, skipping.
'3. Wind Direction' not found, skipping.
'3. Wind Speed/Force' not found, skipping.
'3. Sea State' not found, skipping.
'3. Cloud Cover' not found, skipping.
'3. Weather' not found, skipping.


## 2. Initial Cleanup

### 2.1 Drop Test Entries & Problem Logbooks
Dropping test entries and logbooks which need extensive corrections.

In [5]:
#drop test logbooks and logbooks that need further in-person inspection
standard_drops = ['TEST LOG BOOK NAME', '55', 'Westward-1978', 'TEST TEST HG JULY 2025']
temp_drops = ['Alaska (Bark) 1880-1884', 'Mercator(Ship) 1840-1843','President (Bark) 1865-1869', 'Lancer (Ship) 1865-1868', 
              'Herald (Ship)  1834-1837',  'Lapwing (Ship) 1860-1863', 'Scotland (Bark) 1857-1860']

In [6]:
df1 = trimmed_df.copy()

#always run
for logbook in standard_drops:
    # hello
    # goodbye
    df1.drop(df1.loc[df1["LogBook ID"] == logbook].index,axis=0,inplace=True)

#run until logbooks have been fixed in the database
for logbook in temp_drops:
    df1.drop(df1.loc[df1["LogBook ID"] == logbook].index,axis=0,inplace=True)

### 2.2 DateTime Formatting

In [7]:
# replace DateTime-strings that end with ' nan' with np.nan
df_dt = df1.copy()
df_dt.loc[df_dt.DateTime.astype(str).str.endswith(' nan'), 'DateTime'] = np.nan

In [8]:
# converting 'DateTime' column to actual DateTime and calling it "Entry Date Time"
df_dt['Entry Date Time'] = pd.to_datetime(df_dt.DateTime, format = '%Y-%m-%d %H:%M:%S')
# deleting row "DateTime"
df_dt.drop('DateTime',axis=1)

,ID,LogBook ID,Bottom,Cloud Cover,Depth,Depth Unit,Entry Date,Landmark,Latitude,Local Time,Longitude,Page,Sea State,Ship Heading/Course,Ship Sightings,Wind Direction,Wind Speed/Force,Entry Date Time
0,494,Alpha (ship) 1855-1859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,548,Alpha (ship) 1855-1859,NaN,NaN,NaN,NaN,1855-11-30,NaN,NaN,12:00:00,NaN,30,NaN,E SE,NaN,From SW,Strong Breeze,1855-11-30 12:00:00
3,555,Alpha (ship) 1855-1859,NaN,NaN,NaN,Fathoms,1855-07-14,NaN,NaN,12:00:00,NaN,5,NaN,W by N by the wind,NaN,From S SW,Fine Breeze,1855-07-14 12:00:00
4,556,Alpha (ship) 1855-1859,NaN,NaN,NaN,Fathoms,1855-07-15,NaN,NaN,12:00:00,NaN,5,NaN,SE by the wind,NaN,E NE,NaN,1855-07-15 12:00:00
5,566,Alpha (ship) 1855-1859,NaN,NaN,NaN,NaN,1855-07-25,NaN,NaN,12:00:00,NaN,7,NaN,SW by S,NaN,From SE by S,Moderate Breeze,1855-07-25 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152505,157102,Samuel RobertsonÂ (Ship) 1837-1840,NaN,NaN,NaN,NaN,1838-08-20,NaN,NaN,12:00:00,NaN,59,NaN,E,NaN,SSE,Light winds,1838-08-20 12:00:00
152506,157104,Samuel RobertsonÂ (Ship) 1837-1840,NaN,NaN,NaN,NaN,1838-08-21,NaN,NaN,12:00:00,NaN,59,NaN,S by E,NaN,E by S,Moderate winds,1838-08-21 12:00:00
152507,157105,Samuel RobertsonÂ (Ship) 1837-1840,NaN,NaN,NaN,NaN,1838-08-22,NaN,27 00 S,12:00:00,NaN,60,NaN,SE,NaN,ENE,Moderate winds,1838-08-22 12:00:00
152508,157106,Samuel RobertsonÂ (Ship) 1837-1840,NaN,NaN,NaN,NaN,1838-08-23,NaN,NaN,12:00:00,NaN,60,NaN,SE,NaN,NE by E,Fresh winds,1838-08-23 12:00:00


## 3. Remove Duplicate Entries

### 3.1 Known Duplicate Logbooks
We have a few ships with multiple logbooks as well as duplicate entries that have been verified as intentional (crossing the meridian, noting specific weather, etc.). Here we keep a running list of these so that we only flag new duplicate entries with each new export.

In [9]:
df_no_known_dups = df_dt.copy()

# Removing potential duplicates of data from the Leonidas ships
# Filter rows corresponding to "Leonidas (ship) Journal 1830-1833" and "Leonidas (ship) 1830-1833"
leonidas_journal_rows = df_no_known_dups[df_no_known_dups['LogBook ID'] == 'Leonidas (ship) Journal 1830-1…']
leonidas_rows = df_no_known_dups[df_no_known_dups['LogBook ID'] == 'Leonidas (ship) 1830-1833']

# Get the 'Entry Date Time' values from "Leonidas (ship) Journal 1830-1833"
journal_entry_times = leonidas_journal_rows['Entry Date Time']

# Find indices of rows from "Leonidas (ship) Journal 1830-1833" that have the same 'Entry Date Time' values as in "Leonidas (ship) 1830-1833"
indices_to_drop = leonidas_journal_rows[leonidas_journal_rows['Entry Date Time'].isin(leonidas_rows['Entry Date Time'])].index

# Drop the "Leonidas (ship) Journal 1830-1833" rows from the original DataFrame
df_no_known_dups = df_no_known_dups.drop(indices_to_drop)

In [10]:
# Removing potential duplicates of data from the Margaret ships

# Filter rows corresponding to "Margaret (ship) 1835-1836"
margaret_1835_1836_rows = df_no_known_dups[df_no_known_dups['LogBook ID'] == 'Margaret (ship) 1835-1836']

# Drop rows corresponding to "Margaret (ship) 1835-1836"
df_no_known_dups = df_no_known_dups.drop(margaret_1835_1836_rows.index)

# Filter rows corresponding to "Margaret (ship) 1835-1838" and "Margaret (Ship) 1835–1838"
margaret_ship_1835_1838_rows = df_no_known_dups[df_no_known_dups['LogBook ID'] == 'Margaret (ship) 1835-1838']
margaret_Ship_1835_1838_rows = df_no_known_dups[df_no_known_dups['LogBook ID'] == 'Margaret (Ship) 1835–1838']

# Find indices of rows from "Margaret (ship) 1835-1838" that have the same 'Entry Date Time' values as in "Margaret (Ship) 1835–1838"
indices_to_drop = margaret_ship_1835_1838_rows[margaret_ship_1835_1838_rows['Entry Date Time'].isin(margaret_Ship_1835_1838_rows['Entry Date Time'])].index

# Drop the rows from "Margaret (ship) 1835-1838" that have the same 'Entry Date Time' value as "Margaret (Ship) 1835–1838"
df_no_known_dups = df_no_known_dups.drop(indices_to_drop)

# Rename remaining entries from 'Margaret (ship) 1835-1838' to match 'Margaret (Ship) 1835-1838'
df_no_known_dups['LogBook ID'] = df_no_known_dups['LogBook ID'].replace('Margaret (ship) 1835-1838', 'Margaret (Ship) 1835-1838')
#df

# This should get rid of all entries from Margaret 1835-1836 and then keep rows from
# Margaret (ship) where there are dates missing for Margaret (Ship) and delete the rest

### 3.2 Double Dates

In [11]:
#Reading in ok duplicate entries file:
ok_dups = os.path.join(current_directory, 'permanent_txt_files/ok_duplicate_dates.txt')
ok_df = pd.read_csv(ok_dups, sep="\t")
ok_df['Date'] = pd.to_datetime(ok_df['Date']).dt.date  # ensure 'Date' column is datetime.date objects
ok_set = set(zip(ok_df['LogBook ID'], ok_df['Date']))  # create lookup set
display(ok_set)

{('American (Ship) 1841-1845', datetime.date(1842, 9, 26)),
 ('Atlantic (bark) 1865-1868', datetime.date(1868, 1, 4)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1860, 6, 27)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 7)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 8)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 9)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 10)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 11)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 12)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 13)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 14)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 15)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 16)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 17)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 18)),
 ('Barnstable (Bark) 1860-1864', datetime.date(1862, 7, 19)),
 ('Barnstable (B

In [12]:
#Inspect dates that are indentical in same logbooks
#ie the date was entered incorrectly such that we have entries for 1/1/, 1/2, 1/2, 1/4
#or same day mistakenly entered 2x

df = df_no_known_dups.copy()

grp_cols = ['LogBook ID', 'Entry Date']
groups = df.groupby(grp_cols, dropna=False)
duplicate_date_groups = groups.filter(lambda group: len(group) > 1)

# Collect duplicate ID pairs
duplicate_pairs = []

# Group and check for new duplicates
for (logbook_id, date), group in duplicate_date_groups.groupby(['LogBook ID', duplicate_date_groups['Entry Date Time'].dt.date]):
    
    # SKIP if this duplicate is already approved
    if (logbook_id, date) in ok_set:
        continue

    if logbook_id == 'Canton (ship) 1862-1866':
        continue

    ids = group['ID'].unique()
    if len(ids) == 2:  # Only include clean duplicates (exactly 2 entries)
        duplicate_pairs.append(list(ids))
        print(f"Date: {date}, LogBook ID: {logbook_id}, IDs: {ids}")
    else:
        print(f" Skipping group with {len(ids)} entries on {date} ({logbook_id})")

print(f"Total duplicate pairs found: {len(duplicate_pairs)}")

Date: 1852-01-22, LogBook ID: Chili (Ship) Captain's Journal 1848-1852, IDs: [149731 149732]
Date: 1852-02-24, LogBook ID: Chili (Ship) Captain's Journal 1848-1852, IDs: [149178 149812]
Date: 1837-10-16, LogBook ID: General Jackson (ship) 1836-1839, IDs: [86104 86105]
Date: 1844-01-15, LogBook ID: Harbinger (Ship)  1842-1844, IDs: [144185 144776]
Date: 1843-03-13, LogBook ID: Hibernia (Ship) 1842-1844, IDs: [136933 136934]
Date: 1843-07-14, LogBook ID: Hibernia (Ship) 1842-1844, IDs: [137553 137554]
Date: 1843-09-25, LogBook ID: Hibernia (Ship) 1842-1844, IDs: [137627 137628]
Date: 1840-01-14, LogBook ID: Jasper (ship) 1839-1841, IDs: [129689 132553]
Date: 1840-01-15, LogBook ID: Jasper (ship) 1839-1841, IDs: [129690 132554]
Date: 1840-01-16, LogBook ID: Jasper (ship) 1839-1841, IDs: [129691 132555]
Date: 1840-01-17, LogBook ID: Jasper (ship) 1839-1841, IDs: [129692 132556]
Date: 1840-01-18, LogBook ID: Jasper (ship) 1839-1841, IDs: [129693 132557]
Date: 1840-01-19, LogBook ID: Jasper 

In [13]:
df.loc[df['ID'] == 140506, 'Entry Date'] = '1838-08-28'

In [14]:
import pandas as pd

# 1. Define the IDs to be dropped
ids_to_drop = [137554, 137628, 143849]

# 2. Define the ID-to-Date mapping for updates
date_updates = {
    144776: '1844-02-15',
    132553: '1841-01-14',
    132554: '1841-01-15',
    132555: '1841-01-16',
    132556: '1841-01-17',
    132557: '1841-01-18',
    132558: '1841-01-19',
    132559: '1841-01-20',
    132560: '1841-01-21',
    132561: '1841-01-22',
    132562: '1841-01-23',
    132563: '1841-01-24',
    132564: '1841-01-25',
    128699: '1869-07-19',
    140506: '1838-08-28'
}

# --- Apply Changes ---

# Drop the specified rows (assuming 'ID' is a column)
df = df[~df['ID'].isin(ids_to_drop)].copy()

# Update the 'Entry Date' for the specified IDs
# Using map() with fillna() ensures only specified IDs are changed
df['Entry Date'] = df['ID'].map(date_updates).fillna(df['Entry Date'])

# Optional: Convert to datetime objects if you haven't already
# df['Entry Date'] = pd.to_datetime(df['Entry Date'])

display(f"Cleanup complete. Dropped {len(ids_to_drop)} rows and updated {len(date_updates)} dates.")

'Cleanup complete. Dropped 3 rows and updated 15 dates.'

In [15]:
import pandas as pd

# 1. Ensure Entry Date is a string (to combine with time)
# If it's already a datetime object, this ensures it's in YYYY-MM-DD format
dates = pd.to_datetime(df['Entry Date']).dt.strftime('%Y-%m-%d')

# 2. Extract the Time component from the current (incorrect) DateTime column
# This preserves the 12:00:00 or whatever specific time was recorded
times = pd.to_datetime(df['DateTime']).dt.strftime('%H:%M:%S')

# 3. Combine them into the new corrected strings
corrected_combined = dates + ' ' + times

# 4. Overwrite the columns with the new datetime objects
df['DateTime'] = pd.to_datetime(corrected_combined)
df['Entry Date Time'] = pd.to_datetime(corrected_combined)

# Verification check
print(df[df['ID'] == 132553][['ID', 'Entry Date', 'DateTime', 'Entry Date Time']])

            ID  Entry Date            DateTime     Entry Date Time
125163  132553  1841-01-14 1841-01-14 12:00:00 1841-01-14 12:00:00


#### 3.2.1 New duplicates examined and fixed:

In [25]:
# Define log file location
log_file = "duplicate_corrections_log.txt"
log_path = os.path.join(Files, log_file)

# Reset log at start
with open(log_path, "w") as f:
    f.write("=== Log Start ===\n")

In [28]:
from utils.cleaning import correct_dups

for i, pair in enumerate(duplicate_pairs, 1):
    if pair[0] in date_updates.keys() or pair[1] in date_updates.keys():
        print(f"{pair} already fixed, skipping")
        continue
    clear_output(wait=True)
    print(f"Reviewing {i} of {len(duplicate_pairs)}: IDs {pair}")
    df = correct_dups(df, pair, log_path=log_path)

Reviewing 53 of 53: IDs [np.int64(150213), np.int64(150809)]
Could not find both IDs: [np.int64(150213), np.int64(150809)]


In [29]:
# saving corrected df so we dont have to do that again.
outpath = os.path.join(CSV, "logbook_no_dupes.csv")
df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## 4. Clean Text Columns

### 4.1 Page, Depth, and Ship Sightings

In [30]:
import os, pandas as pd, numpy as np
current_directory = os.getcwd()
CSV = os.path.join(current_directory, 'csv_files')
perm_txt = os.path.join(current_directory, 'permanent_txt_files')
out_txt = os.path.join(current_directory, 'output_txt_files')

df_path = os.path.join(CSV, "logbook_no_dupes.csv")
df1 = pd.read_csv(df_path, low_memory=False)

In [31]:
#Revised version that avoids deprecation:
# Strip strings and store
page_stripped = df1['Page'].astype(str).str.strip()

# Remove brackets based on condition
df1.loc[page_stripped.str.startswith('('), 'Page'] = page_stripped.str[1:4]
df1.loc[page_stripped.str.endswith(')'), 'Page'] = page_stripped.str[0:1]

In [32]:
#Revised future-proofed version
with pd.option_context('display.max_rows', None):
    page_stripped = df1['Page'].astype(str).str.strip()

In [34]:
from utils.cleaning import clean_page_column

temp_df = df1.copy()
df_page_cleaned = clean_page_column(temp_df, column='Page')


4 unhandled values (add to page_map or page_nan):
  - '20-21'
  - '24-25'
  - '29-30'
  - '35-36'


In [35]:
from utils.cleaning import clean_depth_column

temp_df = df1.copy()
df_dep_cleaned = clean_depth_column(temp_df, column='Depth')


5 unhandled values (add to depth_map or depth_nan):
  - '27 & 30'
  - '55?'
  - '6-8'
  - '@ 3am 58, @ 7am 52'
  - 'at 4pm 80, at 6pm 52'


### 4.2 Common String Cleanup
Setting common non-data strings (e.g. "none", "not recorded") to NaN across remaining columns.

In [23]:
#New df to continue cleaning
df2 = df_dep_cleaned.copy()
keywords = ['none', 'nnone', 'not recorded', 'Not Given']

for vname in ['Ship Heading/Course', 'Wind Direction', 'Wind Speed/Force']:
    # Convert to string to handle NaNs safely
    col = df2[vname].astype(str).str.strip()

    for kw in keywords:
        mask = col.str.startswith(kw)
        df2.loc[mask, vname] = np.nan

In [24]:
# removing double spaces from 'Ship Sightings', 'Misc', 'Landmarks', and 'Wind Direction' to make sure rows are sorted properly
# Create a unique separator that does not appear in data
separator = '###'

# Replace consecutive spaces with the separator 
df2['Ship Sightings'] = df2['Ship Sightings'].str.replace(r'\s+', separator, regex=True)
df2['Miscellaneous Observations'] = df2['Miscellaneous Observations'].str.replace(r'\s+', separator, regex=True)
df2['Landmark'] = df2['Landmark'].str.replace(r'\s+', separator, regex=True)

# Replace the separator with a single space
df2['Ship Sightings'] = df2['Ship Sightings'].str.replace(separator, ' ')
df2['Miscellaneous Observations'] = df2['Miscellaneous Observations'].str.replace(separator, ' ')
df2['Landmark'] = df2['Landmark'].str.replace(separator, ' ')

KeyError: 'Miscellaneous Observations'

## 5. Clean Wind Data

### 5.1 Wind Direction

In [ ]:
# Get unique values
unique_directions = df2['Wind Direction'].dropna().unique()

# Save to a text file (one entry per line)
with open('output_txt_files/unique_wind_directions.txt', 'w') as f:
    for direction in unique_directions:
        f.write(f"{direction}\n")

Identifying wind speed terms in the "Wind Direction" column and moving them to "Wind Speed/Force" and "Wind Direction" accordingly.

In [ ]:
df2.loc[df2['Wind Direction']== 'Calm',"Wind Speed/Force"]    # No need to update wind speed entry
df2.loc[df2['Wind Direction']== 'Light',"Wind Speed/Force"] = 'Light'
df2.loc[df2['Wind Direction']== 'Light airs',"Wind Speed/Force"] = 'Light airs'
df2.loc[df2['Wind Direction']== 'light wind',"Wind Speed/Force"] = 'light wind'
df2.loc[df2['Wind Direction']== 'light wind',"Direction"] = 'from NE'

In [ ]:
from utils.cleaning import clean_wind_dirs

In [ ]:
#apply to all wind cleaning to entire dataframe

temp_df = df2.copy()
wind_cleaned_df, to_nan_vals = clean_wind_dirs(temp_df)

#inspect entries set to NaN and add any that should not be to above mapping dicts
print(f"There are {len(to_nan_vals)} values set to NaN")
#to_nan_vals

In [ ]:
from utils.cleaning import wind_dir_to_numeric

temp_df = wind_cleaned_df.copy()
numeric_wind_df = wind_dir_to_numeric(temp_df, col='Clean Wind Direction', out_col='WD_Bearing')

#numeric_wind_df

### 5.2 Wind Speed/Force

In [ ]:
#applying initial string cleaning
from utils.cleaning import init_wind_force_clean

basic_clean_wf_df = init_wind_force_clean(numeric_wind_df)

Converting wind force terms to the Beaufort scale using the classified term map (`permanent_txt_files/wind_force_classified.txt`).

In [ ]:
from utils.cleaning import load_beaufort_map, parse_beaufort_series

wf_txt_file = os.path.join(perm_txt, "wind_force_classified.txt")
log = os.path.join(out_txt, "beaufort_additions_log.txt")

### Load BF mapping from your txt file
bf_map = load_beaufort_map(wf_txt_file, unique_only=True)

# Apply to your DataFrame
outdf = parse_beaufort_series(
    basic_clean_wf_df, 
    col="Wind Speed/Force",
    bf_map=bf_map,
    new_col='BF Value',
    mapping_txt_file=wf_txt_file,   # this will overwrite with updated sections!
    interactive=True,
    log_file=log
)

### 5.3 Sea State, Cloud Cover, and Weather

In [ ]:
from utils.cleaning import clean_remaining_strings
import csv

temp_df = outdf.copy()
all_strings_cleaned_df = clean_remaining_strings(temp_df)

outpath = os.path.join(CSV, "logbooks_clean_strings.csv")
all_strings_cleaned_df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## 6. Coordinate Cleaning

### 6.1 Format Normalization & Conversion

Identifying and correcting:
- Lat/lon values with no direction
- Lat/lon with values larger than 90/180
- Lat/lon entries switched
- Lat with E/W (rather than N/S)
- Lon with S/N (rather than E/W)
- Lat/lon with "Miles" or "M" in them
- Other symbols/formats (e.g. DD.MM)

In [ ]:
# import utilities and define paths
from utils.cleaning import (
    normalize_coords,
    flag_and_convert_miles,
    flag_coords_too_many_digits,
    correct_coord,
    flag_coords_missing_direction,
    flag_direction_symbol_errors,
    batch_correct_coords,
    apply_coord_corrections,
    flag_coords_beyond_bounds,
    examine_and_correct_outliers,
    save_invalid_coords,
)

import os
import pandas as pd

current_directory = os.getcwd()
Figures = os.path.join(current_directory, 'manuscript_figures')
Files   = os.path.join(current_directory, 'output_txt_files')
CSV     = os.path.join(current_directory, 'csv_files')
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

In [ ]:
# load / prepare
df_path = os.path.join(CSV, "logbooks_clean_strings.csv")
df = pd.read_csv(df_path, low_memory=False)

# ensure Entry Date is datetime
if not pd.api.types.is_datetime64_any_dtype(df['Entry Date']):
    df['Entry Date'] = pd.to_datetime(df['Entry Date'], errors='coerce')

# sort for nice context windows
df = df.sort_values(['LogBook ID', 'Entry Date']).reset_index(drop=True)

In [ ]:
# normalize formatting and convert text to DMS
temp_df = df.copy()
file = os.path.join(Files, "coords_normalized.txt")

df_1 = normalize_coords(temp_df, lat_col='Latitude', lon_col='Longitude', verbose=False, save_fixes_file=file)

In [ ]:
# convert special text like Miles/Equator/etc
df_2 = flag_and_convert_miles(df_1)

In [ ]:
# correct entries with too-many-digits
df_3 = batch_correct_coords(df_2, flag_coords_too_many_digits)

In [ ]:
# correct entries missing direction or wrong symbol
outpath = os.path.join(Files, "coord_corrections_log.txt")

# apply known manual corrections (lon > 180 recorded style, etc.)
COORD_FIXES = {
    # Good Return (1841–1844)
    48751: {"Longitude": "148 38 W"}, 48749: {"Longitude": "148 10 W"}, 48747: {"Longitude": "148 55 W"},
    48744: {"Longitude": "148 45 W"}, 48743: {"Longitude": "148 45 W"}, 48740: {"Longitude": "149 00 W"},
    48739: {"Longitude": "149 30 W"}, 48738: {"Longitude": "150 45 W"}, 48737: {"Longitude": "153 00 W"},
    48736: {"Longitude": "153 15 W"}, 48735: {"Longitude": "153 35 W"}, 48734: {"Longitude": "153 30 W"},
    48733: {"Longitude": "155 00 W"}, 48732: {"Longitude": "157 00 W"}, 48731: {"Longitude": "159 20 W"},
    48724: {"Longitude": "161 30 W"}, 48721: {"Longitude": "161 30 W"}, 48714: {"Longitude": "162 52 W"},
    48712: {"Longitude": "164 46 W"}, 48709: {"Longitude": "167 30 W"}, 48708: {"Longitude": "170 30 W"},
    48703: {"Longitude": "173 00 W"}, 48699: {"Longitude": "176 30 W"}, 48697: {"Longitude": "179 00 W"},

    # Gideon Howland (Ship) 1838–1842
    102329: {"Longitude": "170 13 W"}, 102163: {"Longitude": "155 20 W"}, 102161: {"Longitude": "159 00 W"},
    102160: {"Longitude": "161 00 W"}, 102159: {"Longitude": "161 13 W"}, 102158: {"Longitude": "163 33 W"},
    102157: {"Longitude": "165 13 W"}, 102156: {"Longitude": "166 32 W"}, 102155: {"Longitude": "168 04 W"},
    102154: {"Longitude": "169 02 W"}, 102153: {"Longitude": "168 09 W"}, 102152: {"Longitude": "168 14 W"},
    102151: {"Longitude": "169 12 W"}, 102150: {"Longitude": "168 46 W"}, 102149: {"Longitude": "168 48 W"},
    102148: {"Longitude": "168 42 W"}, 102144: {"Longitude": "171 12 W"}, 102143: {"Longitude": "171 18 W"},
    102142: {"Longitude": "171 28 W"}, 102134: {"Longitude": "170 26 W"}, 102131: {"Longitude": "170 59 W"},
    102125: {"Longitude": "177 17 W"}, 102114: {"Longitude": "171 58 W"}, 102111: {"Longitude": "171 36 W"},
    102108: {"Longitude": "172 49 W"}, 102102: {"Longitude": "173 19 W"}, 102100: {"Longitude": "174 15 W"},
    101999: {"Longitude": "174 15 W"}, 101997: {"Longitude": "175 51 W"}, 101953: {"Longitude": "177 13 W"},
    101950: {"Longitude": "178 51 W"}, 101947: {"Longitude": "179 51 W"},

    # President (ship)
    113588: {"Longitude": "176 82 W"},
}

df_4 = apply_coord_corrections(df_3, COORD_FIXES)

In [ ]:
# flag out-of-bounds entrues and correct
flagged_ids_lon, flagged_ids_lat = flag_coords_beyond_bounds(df_4)

df_5 = examine_and_correct_outliers(df_4, flagged_ids_lon, col='Longitude', log_path=outpath)
df_6 = examine_and_correct_outliers(df_5, flagged_ids_lat, col='Latitude',  log_path=outpath)

In [ ]:
from utils.cleaning import final_coord_cleanup

df_7 = final_coord_cleanup(df_6, lat_col='Latitude', lon_col='Longitude')

# convert string "nan" (any case) to real NaN
df_8 = df_7.replace(r'^\s*nan\s*$', np.nan, regex=True)

# output remaining invalid tokens for inspection
invalid_lon, invalid_lat = save_invalid_coords(df_8, dir_path=Files)

In [ ]:
#fix invalids long terms

# Fix the typo '158 00 WW' to '158 00 W'
df_8.loc[df['ID'] == 141451, 'Longitude'] = '158 00 W'

# Fix the typo '1418 55 W' to '148 55 W'
df_8.loc[df['ID'] == 142054, 'Longitude'] = '148 55 W'

# Should be N was entered as W
df_8.loc[df['ID'] == 140516, 'Latitude'] = '16 40 N'

In [ ]:
import csv
outpath = os.path.join(CSV, "logbooks_coords_cleaned.csv")
df_8.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### 6.2 Unrealistic Coordinates
The lat/lon entries have been cleaned, but some inaccuracies often make it through the initial cleaning (due to inaccurate logkeepers or entry error). Here we flag unrealistic coordinate jumps, points over land, and plot the new entries to visually check the ship trajectories.

Identification and flagging in column `coord_diff`.

In [ ]:
import pandas as pd, numpy as np, os

# Get current directory
current_directory = os.getcwd()
#print(current_directory)

# Specify the path to the folder you want to save data to 
Figures = os.path.join(current_directory, 'manuscript_figures')
Files   = os.path.join(current_directory, 'output_txt_files')
CSV     = os.path.join(current_directory, 'csv_files')

# Create directories only if they don't exist
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

In [ ]:
from utils.cleaning import dms_to_decimal, add_decimal_columns, flag_unrealistic_coord_jumps, inspect_and_correct_logbook_flags

df_path = os.path.join(CSV, "logbooks_coords_cleaned.csv")
df = pd.read_csv(df_path, low_memory=False)

In [ ]:
#Drop highly corrupted logbooks

removed_books_df = df.copy()

additional_drops = ['Lapwing (Ship) 1860-1863', 'Barnstable (Bark) 1860-1864', 'Congress (Ship) 1857-1859', 'Jason (Ship)  1846-1848']
for logbook in additional_drops:
    removed_books_df.drop(removed_books_df.loc[removed_books_df["LogBook ID"] == logbook].index,axis=0,inplace=True)

In [ ]:
# add decimal columns to df
df = removed_books_df.copy()

In [ ]:
unflag_list = os.path.join(Files, "unflag_list.txt")
lpath = os.path.join(Files, "coord_corrections_log.txt")

In [ ]:
#Drop decimal columns so that I can recompute
df = df.drop(columns=['Latitude_decimal', 'Longitude_decimal'], errors='ignore')
df = add_decimal_columns(df, lat_col='Latitude', lon_col='Longitude',
                         out_lat='Latitude_decimal', out_lon='Longitude_decimal')

# flag unrealistic jumps
df = flag_unrealistic_coord_jumps(
    df,
    time_col='Entry Date Time',
    logbook_col='LogBook ID',
    lat_col='Latitude_decimal',
    lon_col='Longitude_decimal',
    time_format='%Y-%m-%d %H:%M:%S',      # your current format
    time_delta_seconds=60*60*24*2,        # 2 days
    latlon_delta_deg=10.0,
    lon_delta_upper_limit=320.0
)

# See the flagged rows
suspects = df[df['coord_diff']]

In [ ]:
# Which logbooks actually have flags?
flagged_books = df.loc[
    df['coord_diff']
    & df['Latitude_decimal'].notna()
    & df['Longitude_decimal'].notna(),
    'LogBook ID'
].value_counts()
flagged_books

In [ ]:
logbook_id = flagged_books.index[0]

df = inspect_and_correct_logbook_flags(
    df,
    logbook_id=logbook_id,
    flagged_col="coord_diff",
    annotate=False,
    log_path=lpath,
    skip_list_path=unflag_list,  
    recompute_decimal_after_each=True,
    recompute_decimal_fn=lambda d: d.assign(
        Latitude_decimal=d['Latitude'].apply(dms_to_decimal),
        Longitude_decimal=d['Longitude'].apply(dms_to_decimal)
    ),
    reflag_after_each=True,
    reflag_fn=lambda d: flag_unrealistic_coord_jumps(
        d,
        time_col='Entry Date Time',
        logbook_col='LogBook ID',
        lat_col='Latitude_decimal',
        lon_col='Longitude_decimal',
        time_format='%Y-%m-%d %H:%M:%S',
        time_delta_seconds=60*60*24*2,
        latlon_delta_deg=10.0,
        lon_delta_upper_limit=320.0
    ),
)

In [ ]:
import csv
outpath = os.path.join(CSV, "logbooks_flagged_cleaned.csv")
df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## 7. Visual Inspection

Plotting all new entries that have not been visually cleaned in previous exports to catch any remaining errors.

In [ ]:
#Inspect trajectories of all new points to make sure flagging didnt miss anything

import pandas as pd, os, numpy as np

current_directory = os.getcwd()
Figures = os.path.join(current_directory, 'manuscript_figures')
CSV = os.path.join(current_directory, 'csv_files')
Files = os.path.join(current_directory, 'output_txt_files')

# Load both exports
df_old = pd.read_csv('~/Documents/whaling_logs/20251021_export/csv_files/logentries-export-2025-11-03.csv',  low_memory=False)
df_new = pd.read_csv('~/Documents/whaling_logs/20260211_export/csv_files/logentries-export-2026-02-11.csv', low_memory=False)

df_old['Entry Date'] = pd.to_datetime(df_old['Entry Date'], errors='coerce', format='mixed')
df_new['Entry Date'] = pd.to_datetime(df_new['Entry Date'], errors='coerce', format='mixed')

df_old['Entry Date'] = df_old['Entry Date'].dt.normalize()
df_new['Entry Date'] = df_new['Entry Date'].dt.normalize()

df_old_filled = df_old.fillna('##MISSING##')
df_new_filled = df_new.fillna('##MISSING##')

# Mapping of truncated names in new df -> full names
logbook_overwrite_map = {
    "A. M. Nicholson (schooner) 190…": "A. M. Nicholson (schooner) 1909-1910",
    "Francis Allyn (Schooner) 1891-…": "Francis Allyn (Schooner) 1891-1893",
    "Governor Hopkins (Brig) 1839-1…": "Governor Hopkins (Brig) 1839-1840",
    "George Washington (Bark) 1837-…": "George Washington (Bark) 1837-1840",
    "Gideon Howland (Ship) 1838-184…": "Gideon Howland (Ship) 1838-1842",
    "Gideon Howland (ship) 1836-183…": "Gideon Howland (ship) 1836-1838",
    "Walter Irving (Schooner) 1852-…": "Walter Irving (Schooner) 1852-1853",
    "Walter Irving (Schooner) 1856-…": "Walter Irving (Schooner) 1856-1857",
    "Walter Irving (Schooner) 1855-…": "Walter Irving (Schooner) 1855-1856",
    "General Jackson (ship) 1836-18…": "General Jackson (ship) 1836-1839",
    "Walter Irving (Schooner) 1854-…": "Walter Irving (Schooner) 1854-1855",
    "Walter Irving (Schooner) 1853-…": "Walter Irving (Schooner) 1853-1854",
    #"Gage H. Phillips (Schooner) 18…": "<FILL IN FULL NAME>",  # placeholder
    "Sally Anne (ship) & Seine…": "Sally Anne (ship) & Seine (bark) 1835-1838",
    "C.C. Comstock (Schooner) 1865-…": "C.C. Comstock (Schooner) 1865-1866",
    "George Clinton (ship) 1834-183…": "George Clinton (ship) 1834-1837",
    "Charles Phelps (Ship) 1850-185…": "Charles Phelps (Ship) 1850-1853",
    "Abraham Barker (Ship) 1850-185…": "Abraham Barker (Ship) 1850-1853",
    "Henry Kneeland (ship) 1848-185…": "Henry Kneeland (ship) 1848-1851",
    "Good Return II (ship) 1833-183…": "Good Return II (ship) 1833-1834",
    "Charles and Henry (ship) 1833-…": "Charles and Henry (ship) 1833-1836",
    "Bartholomew Gosnold (Bark) 188…": "Bartholomew Gosnold (Bark) 1881-1885",
    "Leonidas (ship) Journal 1830-1…": "Leonidas (ship) Journal 1830-1833",
    "Eunice H. Adams (Brig) 1887-18…": "Eunice H. Adams (Brig) 1887-1890",
    "Governor Carver (ship) 1857-18…": "Governor Carver (ship) 1857-1859",
    "Charles Phelps (Ship) 1844-184…": "Charles Phelps (Ship) 1844-1847",
    "Clifford Wayne (ship) 1844-184…": "Clifford Wayne (ship) 1844-1847",
    "Charles Phelps (ship) 1842-184…": "Charles Phelps (ship) 1842-1844",
    "Thomas Winslow (Brig) 1846-184…": "Thomas Winslow (Brig) 1846-1847",
    "Charles W. Morgan (bark) 1911-…": "Charles W. Morgan (bark) 1911-1912",
}

# Apply the mapping to overwrite in df_new_filled
df_new_filled["LogBook ID"] = df_new_filled["LogBook ID"].replace(logbook_overwrite_map)
df_old_filled["LogBook ID"] = df_old_filled["LogBook ID"].replace(logbook_overwrite_map)

In [ ]:
# Find new rows
new_rows = df_new_filled.merge(df_old_filled.drop_duplicates(), how='left', indicator=True)
new_rows = new_rows[new_rows['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display result
print(f"{len(new_rows)} new row(s) found.")
#display(new_rows)

In [ ]:
# Extract unique values from the 'LogBook ID' column
logbooks_new_entries = new_rows['LogBook ID'].unique()
logbooks_new_entries = logbooks_new_entries[logbooks_new_entries != "Gage H. Phillips (Schooner) 18…"]

temp_drops = ['Alaska (Bark) 1880-1884', 'Mercator(Ship) 1840-1843','President (Bark) 1865-1869', 
              'Lancer (Ship) 1865-1868', 'Herald (Ship)  1834-1837',  'Lapwing (Ship) 1860-1863']
logbooks_new_entries = [x for x in logbooks_new_entries if x not in temp_drops]

# Convert to a sorted list (optional)
logbooks_new_entries = sorted(logbooks_new_entries)

# Display result
print(f"{len(logbooks_new_entries)} logbooks with new entries found:")
for logbook in logbooks_new_entries:
    print(logbook)

In [ ]:
# Get sets of unique logbook IDs
new_logbook_ids = set(df_new_filled['LogBook ID'].unique())
old_logbook_ids = set(df_old_filled['LogBook ID'].unique())

# Subtract to find newly added logbooks
newly_added_logbooks = sorted(new_logbook_ids - old_logbook_ids)

# Display the result
print(f"{len(newly_added_logbooks)} new logbooks found:")
for logbook in newly_added_logbooks:
    print(logbook)

In [ ]:
df_path = os.path.join(CSV, "logbooks_flagged_cleaned.csv")
df_flagged = pd.read_csv(df_path, low_memory=False)

# apply previously made corrections to avoid repeating in exploratory code
COORD_FIXES = {
    # Amazon (1856–1860)
    125836: {"Latitude": "0 30 S"},
    125837: {"Latitude": "0 52 S"},
    125839: {"Latitude": "1 48 S"},
    125841: {"Latitude": "2 31 S"},
    125843: {"Latitude": "3 50 S"},
    125844: {"Latitude": "4 57 S"},

    # Scotland (Bark) 1857–1860
    136891: {"Longitude": "62 16 W"}, 

    # Falcon (Bark) 1865–1867
    119191: {"Latitude": "nan"},  # as given

    # Elisha Dunbar (Bark) 1854–1858
    130023: {"Longitude": "112 02 E"},

    # Lapwing (Ship) 1860–1863
    132504: {"Longitude": "108 10 W"},

    # Barnstable (Bark) 1860–1864
    134718: {"Latitude": "nan"},

    # Jasper (Bark) 1840–1842
    134162: {"Longitude": "39 57 W"},
    134163: {"Longitude": "39 40 W"},

    # Pioneer (Bark) 1858–1861
    130660: {"Longitude": "149 30 W"},

    # Young Phenix (1867–1871)
    122652: {"Longitude": "125 34 E"},

    # Amethyst (Ship) 1838–1840
    136970: {"Longitude": "172 23 E"},
    134846: {"Latitude": "38 29 N"},
}

from utils.cleaning import apply_coord_corrections, dms_to_decimal

df = apply_coord_corrections(df_flagged, COORD_FIXES)

df['Latitude_decimal']  = df['Latitude'].apply(dms_to_decimal)
df['Longitude_decimal'] = df['Longitude'].apply(dms_to_decimal)

In [ ]:
#some hardcoded changes that we have made within the db but need to inlclude in our export
# target dates for the Pioneer logbook
target_dates = ['1878-01-22', '1878-01-23', '1878-01-24', '1878-01-25', '1878-01-26']
log_id = 'Pioneer (Bark) 1877-1880'

# replace 'W' with 'E' in the string column for those rows
df.loc[
    (df['LogBook ID'] == log_id) & 
    (df['Entry Date'].isin(target_dates)), 
    'Longitude'
] = df['Longitude'].str.replace('W', 'E', case=False)

# Brandt (Ship) 1838-1839: Change lon from E to W for 1838-11-20
df.loc[(df['LogBook ID'] == 'Brandt (Ship) 1838-1839') & (df['Entry Date'] == '1838-11-20'), 'Longitude'] = \
    df['Longitude'].str.replace('E', 'W', case=False)

# Hibernia (Ship) 1844-1846: Change lat from S to N for 1845-05-06
df.loc[(df['LogBook ID'] == 'Hibernia (Ship) 1844-1846') & (df['Entry Date'] == '1845-05-06'), 'Latitude'] = \
    df['Latitude'].str.replace('S', 'N', case=False)

df['Latitude_decimal']  = df['Latitude'].apply(dms_to_decimal)
df['Longitude_decimal'] = df['Longitude'].apply(dms_to_decimal)

In [ ]:
from utils.cleaning import plot_new_entries

fig, axes, info = plot_new_entries(
    df_corrected=df,
    new_rows=new_rows,
    figures_dir= Figures, #set to Figures to save
    export_label="2025-08-12 export",  
    plot_scope="all_from_new_logbooks",   
    ncols=2,
    exclude_logbooks=[
        "Gage H. Phillips (Schooner) 18…", "TEST TEST HG JULY 2025"  # exclusions logbook that isn't new
    ],
)
print(info)

In [ ]:
ship_name = 'Young Phoenix (Ship) 1860-1863'

# Convert ALL relevant columns to datetime first
df['Entry Date'] = pd.to_datetime(df['Entry Date'])
df['Entry Date Time'] = pd.to_datetime(df['Entry Date Time'])
df['DateTime'] = pd.to_datetime(df['DateTime'])

#  Create mask based on one of the corrected datetime columns
mask = (df['LogBook ID'] == ship_name) & (df['Entry Date'].dt.year >= 1900)

#  View entries (Optional check)
print(f"Found {mask.sum()} entries to fix for {ship_name}:")

#  Apply the 100-year offset (Now works because all columns are datetime objects)
df.loc[mask, 'Entry Date'] = df.loc[mask, 'Entry Date'] - pd.DateOffset(years=100)
df.loc[mask, 'Entry Date Time'] = df.loc[mask, 'Entry Date Time'] - pd.DateOffset(years=100)
df.loc[mask, 'DateTime'] = df.loc[mask, 'DateTime'] - pd.DateOffset(years=100)

# Quick Verification
print("\nVerification - New date range for Young Phoenix:")
print(df[df['LogBook ID'] == ship_name]['Entry Date'].dt.year.unique())

In [ ]:
df[df['LogBook ID'] == ship_name].columns

In [ ]:
from utils.cleaning import plot_logbook

logbook = "Pioneer (Bark) 1877-1880"
logbook_name = logbook.split('(')[0]

_ = plot_logbook(
    df, logbook,
    # years=[1858],                
    annotate=False,                
    annotate_field="Entry Date",
    annotate_max=800,  
    figures_dir = Figures,
    filename=f"inspect_{logbook_name}.png",
    dpi=300,
)

In [ ]:
#apply corrections as needed

from utils.cleaning import correct_coord
from utils.cleaning import dms_to_decimal

lpath = os.path.join(Files, "coord_corrections_log.txt")
#correct_coord(df, idx, col = 'Longitude', log_path=lpath, force_both=False)
#correct_coord(df, idx, col = 'Longitude', log_path=lpath, force_both=False)


# recompute decimal degrees for adjusted row
# df.loc[idx, 'Latitude_decimal']  = dms_to_decimal(df.loc[idx, 'Latitude'])
# df.loc[idx, 'Longitude_decimal'] = dms_to_decimal(df.loc[idx, 'Longitude'])

## 8. Export & Final Standardization

### 8.1 Save Cleaned Dataset
After inspecting all new entries and making corrections, save the cleaned dataset.

In [ ]:
# Work with a copy to avoid SettingWithCopyWarning
temp_df = df.copy()

# Convert to datetime safely
temp_df['Entry Date Time'] = pd.to_datetime(temp_df['Entry Date Time'], errors='coerce')

# List of logbooks to exclude completely
exclude_logbooks = [
    'Scotland (Bark) 1857-1860',
    'Mercator(Ship) 1840-1843',
    # "TEST TEST HG JULY 2025"
]

# Define rows to exclude
exclude_rows = (
    (temp_df['LogBook ID'].isin(exclude_logbooks))  
)

# Apply filtering and save
current_directory = os.getcwd()
PKL = os.path.join(current_directory, "pkl_files")
os.makedirs(PKL, exist_ok=True)

df_cleaned = temp_df[~exclude_rows]
df_cleaned.to_csv(f'{CSV}/final_logbooks.csv', index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
df_cleaned.to_pickle(f"{PKL}/Tier1.pkl")

### 8.2 Standardize Beaufort Classifications
Map multi-category Beaufort values (e.g. 23, 45) to single integer values.

In [ ]:
df = pd.read_csv(os.path.join(CSV, 'final_logbooks.csv'), low_memory=False)

unique_bf = df['BF Value'].unique()
print(unique_bf)

In [ ]:
def map_bf(value):
    """
    Map Beaufort values:
    - Keep valid values 0–12 as-is.
    - Map ranges like 23 → 2, 45 → 4, 56 → 5, 67 → 6.
    - Leave NaN as NaN.
    """
    if pd.isna(value):
        return np.nan
    try:
        v = int(value)
        if 0 <= v <= 12:
            return v
        else:
            return int(str(v)[0])  # take the first digit for borderline cases
    except (ValueError, TypeError):
        return np.nan

df['BF Value Mapped'] = df['BF Value'].apply(map_bf)
print(sorted(df['BF Value Mapped'].dropna().unique()))

In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['BF Value'])

df = df.rename(columns={'BF Value Mapped': 'BF Value'})

cols = list(df.columns)
insert_at = cols.index('Wind Speed/Force') + 1
cols.remove('BF Value')
cols.insert(insert_at, 'BF Value')
df = df[cols]

out_df = df.copy()
out_df['Entry Date Time'] = pd.to_datetime(out_df['Entry Date Time'], errors='coerce')

# Save as  CSV
out_base = "Tier1"  
out_df.to_csv(os.path.join(CSV, f"{out_base}.csv"), index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
out_df.to_pickle(os.path.join(PKL, f"{out_base}.pkl"))